# 06_03 - Decision Engine: Policy Evaluation
## 1. Methodology Overview

This notebook evaluates policy outputs with the shared project utilities for action frequency, coverage, reasons, and risk-context alignment.
It compares heuristic and RL decisions on the same validation period to support transparent decision-engine diagnostics.

**Source data used in this notebook:**
- `data/processed/train_features.csv`
- `data/processed/validation_features.csv`

**Core modules used in this notebook:**
- `src/models/train_model.py`
- `src/decision/policy_inputs.py`
- `src/decision/heuristic_policy.py`
- `src/rl/train_rl_agent.py`
- `src/decision/rl_policy.py`
- `src/decision/policy_evaluation.py`

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = Path('../../').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

train_df = pd.read_csv(project_root / 'data/processed/train_features.csv')
validation_df = pd.read_csv(project_root / 'data/processed/validation_features.csv')

print(f'Train features: {train_df.shape[0]} rows x {train_df.shape[1]} columns')
print(f'Validation features: {validation_df.shape[0]} rows x {validation_df.shape[1]} columns')

## 2. Build Heuristic and RL Decisions on a Common Input Set

To evaluate policies fairly, both engines are applied to the same policy-input dataframe derived from quantile forecasts.

In [ ]:
from src.models.train_model import train_quantile_suite
from src.decision.policy_inputs import prepare_policy_inputs
from src.decision.heuristic_policy import apply_heuristic_policy
from src.rl.train_rl_agent import train_q_learning_agent
from src.decision.rl_policy import apply_rl_policy

quantile_output = train_quantile_suite(train_df=train_df, eval_df=validation_df)
policy_inputs_df = prepare_policy_inputs(validation_df, quantile_output.results)

heuristic_policy_df = apply_heuristic_policy(policy_inputs_df)

rl_training_artifacts = train_q_learning_agent(policy_inputs_df)
rl_policy_artifacts = apply_rl_policy(
    agent=rl_training_artifacts.agent,
    policy_inputs_df=policy_inputs_df,
    include_q_values=False,
 )

rl_policy_df = policy_inputs_df.copy().reset_index(drop=True)
rl_policy_df['recommended_action'] = rl_policy_artifacts.decisions_df['recommended_action'].values
rl_policy_df['action_taken'] = rl_policy_df['recommended_action']

## 3. Evaluate Actions, Coverage, and Risk-Signal Alignment

We run the same evaluation helpers on both policy outputs to make diagnostics directly comparable.

In [ ]:
from src.decision.policy_evaluation import (
    summarize_actions_by_calendar_context,
    summarize_actions_vs_risk_signals,
    summarize_policy_actions,
    summarize_policy_reasons,
    summarize_policy_time_coverage,
)

print('Heuristic policy: action summary')
display(summarize_policy_actions(heuristic_policy_df))

print('RL policy: action summary')
display(summarize_policy_actions(rl_policy_df))

print('Heuristic policy: time coverage')
display(summarize_policy_time_coverage(heuristic_policy_df))

print('RL policy: time coverage')
display(summarize_policy_time_coverage(rl_policy_df))

print('Heuristic policy: calendar context')
display(summarize_actions_by_calendar_context(heuristic_policy_df))

print('RL policy: calendar context')
display(summarize_actions_by_calendar_context(rl_policy_df))

print('Heuristic policy: actions vs risk signals')
display(summarize_actions_vs_risk_signals(heuristic_policy_df))

print('RL policy: actions vs risk signals')
display(summarize_actions_vs_risk_signals(rl_policy_df))

print('Heuristic policy: decision reasons')
display(summarize_policy_reasons(heuristic_policy_df))